# Autoregressive decoding and cache reuse


Autoregressive generation factorizes a sequence as $p(x_1,\ldots,x_T)=\prod_t p(x_t|x_{<t})$. This toy model makes repeated prefix work visible; it is not a neural KV cache.

In [ ]:
from functools import lru_cache
import time

calls={"uncached":0,"cached":0}
def score_prefix(prefix):
    calls["uncached"] += 1
    return sum(ord(c) for c in prefix)%97
@lru_cache(maxsize=None)
def cached_score(prefix):
    calls["cached"] += 1
    return sum(ord(c) for c in prefix)%97

prompts=["omni serving"]*20
for p in prompts:
    score_prefix(p)
for p in prompts:
    cached_score(p)
print(calls)
assert calls == {"uncached":20,"cached":1}

Real KV caches store attention keys and values for processed tokens. Prefix caching can additionally reuse shared prompt prefixes across requests. The memory/performance tradeoff is central to AR serving.

In [ ]:
vocab=[" sees"," hears"," thinks"," speaks"]
def generate(prompt, steps=4):
    out=prompt
    for i in range(steps): out += vocab[(len(out)+i)%len(vocab)]
    return out
generate("The omni model")

### Exercise
Change the prompts so only part of the prefix is shared. What cache key would a production server need?